# Data Cleaning and Feature Preparation
This notebook prepares the raw Online Retail dataset for SQL analysis and machine learning. The main goals are to inspect data quality, define cancellation logic, handle unusual invoice cases, create basic features, and export a cleaned dataset.

In [103]:
import pandas as pd

In [104]:
def sp(second_row=True):
    print('='*70)
    if second_row:
        print()
        print('='*70)

pd.set_option('display.width', 150)
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)

In [105]:
df_orig = pd.read_excel('../data/raw/Online Retail.xlsx')

## Initial Dataset Inspection

In [106]:
def get_df_info(df: pd.DataFrame):
    print("Shape:", df.shape)
    print("Columns:", df.columns)
    sp()
    print(df.info())
    sp()
    print(df.head())
    sp()
    print(df.describe())
    sp()
    print(df.isnull().sum())

In [107]:
get_df_info(df_orig)

Shape: (541909, 8)
Columns: Index(['InvoiceNo', 'StockCode', 'Description', 'Quantity', 'InvoiceDate', 'UnitPrice', 'CustomerID', 'Country'], dtype='str')

<class 'pandas.DataFrame'>
RangeIndex: 541909 entries, 0 to 541908
Data columns (total 8 columns):
 #   Column       Non-Null Count   Dtype         
---  ------       --------------   -----         
 0   InvoiceNo    541909 non-null  object        
 1   StockCode    541909 non-null  object        
 2   Description  540455 non-null  object        
 3   Quantity     541909 non-null  int64         
 4   InvoiceDate  541909 non-null  datetime64[us]
 5   UnitPrice    541909 non-null  float64       
 6   CustomerID   406829 non-null  float64       
 7   Country      541909 non-null  str           
dtypes: datetime64[us](1), float64(2), int64(1), object(3), str(1)
memory usage: 33.1+ MB
None

  InvoiceNo StockCode                          Description  Quantity         InvoiceDate  UnitPrice  CustomerID         Country
0    536365    85123A

## Cancellation Encoding Analysis
Since dataset may store information about canceled orders differently, it is required to find out which approach was used and prove it.
1. Each observation represents an independent transaction: an independent group of items (order) that was purchased and either saved or returned.
2. Observation of order that was canceled relates to the items that were already purchased. It means that the new observation with cancellation status has its own `InvoiceNo` and strictly relates to items that were already purchased.

If it is proven that the second approach has contradictions, the first approach will be used.

The main problem of the second approach is that `UnitPrice` and `Quantity` can differ for purchasing and returning (see cases: `customerid = 15810  AND stockcode = '22380'` and `customerid = 12908`). Additionally, the cancellation date can be earlier than the purchasing date (`customerid = 12908`).

It would not be safe to rely on the `InvoiceDate` or the revenue to find a contradiction of the second approach. But it is possible to check how many items users purchased (according to the second approach) for the whole time. If the negative number quantity is received, the second approach cannot be used.

In [108]:
CHECK_SECOND_APPROACH = True

def check_second_approach(df_orig: pd.DataFrame):
    df = df_orig[~df_orig['CustomerID'].isna()].groupby(['CustomerID', 'StockCode'])['Quantity'].sum()
    df_contr = df[df < 0]
    return df_contr.shape[0], df.shape[0]

if CHECK_SECOND_APPROACH:
    contr, total = check_second_approach(df_orig)
    print(contr, 'out of', total, 'cases when users cancelled more items that were ordered.')

892 out of 267615 cases when users cancelled more items that were ordered.


892 cases of contradiction of the second approach were found. It is 0.33% out of the number of groups (Customer, Item) and it is 9.6% out of the number of all observations that were canceled.

It is clear that some `CustomerID` may not be preserved.
That is why it is required to assume that observations with `CustomerID = NaN` could make the sum of ordered items of some customers a nonnegative number. But the case with `CustomerID = 12474` and `StockCode = 22779` shows that adding orders of customers without ID to the negative sum may not make the sum a nonnegative number.

That is why the first approach of cancellation notation will be considered the correct approach.

## Invoice Number Investigation
Dataset has 3 observations with `InvoiceNo` starting neither with a digit nor with `C`. These cases include two out of two cases with `UnitPrice` less than zero. Since such invoice notation was not specified in the dataset description, it will be safer for future analysis to drop such observations.

In [109]:
bit_mask_invoiceno = df_orig['InvoiceNo'].str.startswith('C', na=False)
for digit in range(10):
    bit_mask_invoiceno = bit_mask_invoiceno | df_orig['InvoiceNo'].astype(str).str.startswith(str(digit), na=False)

print(df_orig[~bit_mask_invoiceno])
sp()
print(df_orig[df_orig['UnitPrice'] < 0])

       InvoiceNo StockCode      Description  Quantity         InvoiceDate  UnitPrice  CustomerID         Country
299982   A563185         B  Adjust bad debt         1 2011-08-12 14:50:00   11062.06         NaN  United Kingdom
299983   A563186         B  Adjust bad debt         1 2011-08-12 14:51:00  -11062.06         NaN  United Kingdom
299984   A563187         B  Adjust bad debt         1 2011-08-12 14:52:00  -11062.06         NaN  United Kingdom

       InvoiceNo StockCode      Description  Quantity         InvoiceDate  UnitPrice  CustomerID         Country
299983   A563186         B  Adjust bad debt         1 2011-08-12 14:51:00  -11062.06         NaN  United Kingdom
299984   A563187         B  Adjust bad debt         1 2011-08-12 14:52:00  -11062.06         NaN  United Kingdom


## Cleaning and Feature Engineering

In [110]:
def clean_dataframe(df_orig: pd.DataFrame):
    df = df_orig.copy()

    bit_mask_invoiceno = df_orig['InvoiceNo'].str.startswith('C', na=False)
    for digit in range(10):
        bit_mask_invoiceno = bit_mask_invoiceno | df_orig['InvoiceNo'].astype(str).str.startswith(str(digit), na=False)
    df = df[bit_mask_invoiceno]

    return df

In [111]:
def add_features(df_orig: pd.DataFrame):
    df = df_orig.copy()

    df['Quantity'] = df['Quantity'].abs()

    if ('InvoiceNo' in df.columns):
        df['Canceled'] = df['InvoiceNo'].str.startswith('C', na=False)
    if ('UnitPrice' in df.columns and 'Quantity' in df.columns):
        df['Revenue'] = df['UnitPrice'] * df['Quantity']

    return df

In [112]:
def final_clean_dataframe(df_orig: pd.DataFrame):
    def cleaning_invoiceno(invoiceno: str):
        if str(invoiceno).startswith('C'):
            return int(str(invoiceno)[1:])
        return int(invoiceno)

    df = df_orig.copy()
    df['InvoiceNo'] = df['InvoiceNo'].apply(cleaning_invoiceno)
    df['InvoiceNo'] = df['InvoiceNo'].astype('Int64')
    return df

In [113]:
df = clean_dataframe(df_orig)
df = add_features(df)
df = final_clean_dataframe(df)

## Duplicate Records Check

In [114]:
duplicate_count = df.duplicated().sum()
print(duplicate_count, 'duplicated rows.')

5268 duplicated rows.


## Zero-Revenue Orders Check

In [115]:
zero_values = df[df['Revenue'] == 0].shape[0]
zero_values_canceled = df[(df['Revenue'] == 0) & (df['Canceled'])].shape[0]
print(zero_values, "rows with zero revenue.")
print(zero_values_canceled, "rows with zero revenue and canceled order.")

2515 rows with zero revenue.
0 rows with zero revenue and canceled order.


## Outlier Investigation

In [116]:
print('UnitPrice Outlier')
print(df.sort_values('UnitPrice', ascending=False).head(10))
sp()
print('Quantity Outlier')
print(df.sort_values('Quantity', ascending=False).head(10))
sp()
print('Revenue Outlier')
print(df.sort_values('Revenue', ascending=False).head(10))

UnitPrice Outlier
        InvoiceNo  StockCode Description  Quantity         InvoiceDate  UnitPrice  CustomerID         Country  Canceled   Revenue
222681     556445          M      Manual         1 2011-06-10 15:31:00   38970.00     15098.0  United Kingdom      True  38970.00
524602     580605  AMAZONFEE  AMAZON FEE         1 2011-12-05 11:36:00   17836.46         NaN  United Kingdom      True  17836.46
43702      540117  AMAZONFEE  AMAZON FEE         1 2011-01-05 09:55:00   16888.02         NaN  United Kingdom      True  16888.02
43703      540118  AMAZONFEE  AMAZON FEE         1 2011-01-05 09:57:00   16453.71         NaN  United Kingdom      True  16453.71
15017      537632  AMAZONFEE  AMAZON FEE         1 2010-12-07 15:08:00   13541.33         NaN  United Kingdom     False  13541.33
15016      537630  AMAZONFEE  AMAZON FEE         1 2010-12-07 15:04:00   13541.33         NaN  United Kingdom      True  13541.33
16356      537651  AMAZONFEE  AMAZON FEE         1 2010-12-07 15:49:00  

## Final Cleaned Dataset

In [117]:
get_df_info(df)

Shape: (541906, 10)
Columns: Index(['InvoiceNo', 'StockCode', 'Description', 'Quantity', 'InvoiceDate', 'UnitPrice', 'CustomerID', 'Country', 'Canceled', 'Revenue'], dtype='str')

<class 'pandas.DataFrame'>
Index: 541906 entries, 0 to 541908
Data columns (total 10 columns):
 #   Column       Non-Null Count   Dtype         
---  ------       --------------   -----         
 0   InvoiceNo    541906 non-null  Int64         
 1   StockCode    541906 non-null  object        
 2   Description  540452 non-null  object        
 3   Quantity     541906 non-null  int64         
 4   InvoiceDate  541906 non-null  datetime64[us]
 5   UnitPrice    541906 non-null  float64       
 6   CustomerID   406829 non-null  float64       
 7   Country      541906 non-null  str           
 8   Canceled     541906 non-null  bool          
 9   Revenue      541906 non-null  float64       
dtypes: Int64(1), bool(1), datetime64[us](1), float64(3), int64(1), object(2), str(1)
memory usage: 42.4+ MB
None

   Invoice

## Cleaning Summary

The raw dataset contains 541,909 transaction rows. After investigating invoice notation, cancellation behavior, duplicate rows, zero-revenue records, and extreme values, the dataset was prepared for further SQL analysis and predictive modeling.

Key decisions:
- Cancellation status was extracted from invoice numbers starting with `C`.
- Unspecified invoice types related to bad debt adjustments were removed.
- Negative `Quantity` values were converted to absolute values after cancellation status was preserved.
- Missing `CustomerID` values were retained because they are still useful for product, country, revenue, and cancellation-level analysis.
- Duplicate rows were retained because they may represent repeated product lines within the same invoice.
- Extreme values were investigated and kept because they correspond to identifiable business events.

The cleaned dataset is ready for PostgreSQL loading and further analysis.

## Exporting Data

In [118]:
old_columns = df.columns

df.columns = df.columns.str.lower()

df.to_csv(
    "../data/processed/retail_clean.csv",
    index=False
)

df.columns = old_columns

Running file `src/load_data.py`.